### Statistical Analysis & Understanding Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys


# Import specific functions from utils
from utils import (
    handle_categorical_missing_values,
    handle_numerical_missing_values,
    display_missing_info,
    encode_features,
    scale_features,
    look_for_outliers,
    handle_outliers_iqr,
    treat_skewness,
    apply_pca,


)
from models.naive_bayes import naive_bayes_classifier, naive_bayes_with_grid_search
from sklearn.model_selection import train_test_split

# from google.colab import drive

In [ ]:
# pip install category_encoders

In [ ]:
# drive.mount('/content/drive')

In [ ]:
# Define root
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
sys.path.append(PROJECT_ROOT)

# Load raw data
data_path = os.path.join(PROJECT_ROOT, "data", "train data.csv")
df = pd.read_csv(data_path)

In [ ]:
df.tail()

In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
print(f"Dataset contains {df.shape[0]} rows and {df.shape[1]} columns.")

In [ ]:
#Summarize categorical columns:
for col in df.select_dtypes(include='object').columns:
    print(f"Value counts for {col}:\n{df[col].value_counts()}\n")

In [ ]:
for col in df.columns:
        num_unique = df[col].nunique()
        print(f"Column '{col}' has {num_unique} unique values.")

In [ ]:
print(f"Number of duplicate rows: {df.duplicated().sum()}")
# View the duplicated rows
duplicated_rows = df[df.duplicated()]
print("\nDuplicated rows:")
print(duplicated_rows)

In [ ]:
# 1. Descriptive statistics for numeric columns
numeric_desc = df.describe()
print("Descriptive Statistics (Numeric Columns):")
print(numeric_desc)

In [ ]:
# 2. Frequency distribution for top 10 values in categorical variables
categorical_cols = df.select_dtypes(include='object').columns
print("\nFrequency Distribution (Top 10 per Categorical Column):")
for col in categorical_cols:
    print(f"\nTop values in '{col}':")
    print(df[col].value_counts().head(10))

In [ ]:
# 3. Correlation matrix for numerical features
print("\nCorrelation Matrix:")
correlation_matrix = df.corr(numeric_only=True)
print(correlation_matrix)

In [ ]:
# 4. Distribution plots (Histograms) for numerical columns
numeric_cols = df.select_dtypes(include=[np.number]).columns

plt.figure(figsize=(15, 12))
for i, col in enumerate(numeric_cols, 1):
    plt.subplot(3, 3, i)
    sns.histplot(df[col], kde=True, bins=30)
    plt.title(f'Histogram: {col}')
plt.tight_layout()
plt.show()

In [ ]:
# 5. Boxplots for outlier detection
plt.figure(figsize=(15, 12))
for i, col in enumerate(numeric_cols, 1):
    plt.subplot(3, 3, i)
    sns.boxplot(x=df[col])
    plt.title(f'Boxplot: {col}')
plt.tight_layout()
plt.show()

In [ ]:
# 6. Bar charts for top categories
plt.figure(figsize=(18, 18))
for i, col in enumerate(categorical_cols[:9], 1):  # Limit to 9 for layout
    plt.subplot(3, 3, i)
    df[col].value_counts().head(10).plot(kind='bar')
    plt.title(f'Bar Chart: {col}')
    plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Preprocessing

In [ ]:
display_missing_info(df)

In [ ]:
print(df.columns)

In [ ]:
df = handle_categorical_missing_values(df) 
df = handle_numerical_missing_values(df)

In [ ]:
display_missing_info(df)

In [ ]:
df.isnull().sum()

In [ ]:
# Step 2: Standardize text-based categorical fields

df['Name'] = df['Name'].str.title()
df['Gender'] = df['Gender'].str.capitalize()
df['Medical Condition'] = df['Medical Condition'].str.capitalize()
df['Doctor'] = df['Doctor'].str.title()
df['Hospital'] = df['Hospital'].str.title()
df['Insurance Provider'] = df['Insurance Provider'].str.title()
df['Medication'] = df['Medication'].str.capitalize()
df['Test Results'] = df['Test Results'].str.capitalize()
df['Admission Type'] = df['Admission Type'].str.capitalize()

In [ ]:
# Step 3: Convert date columns to datetime & Calculate Length of Stay
df["Date of Admission"] = pd.to_datetime(
        df["Date of Admission"], errors="coerce", dayfirst=True
    )
df["Discharge Date"] = pd.to_datetime(
    df["Discharge Date"], errors="coerce", dayfirst=True
)

# Calculate Length of Stay
df['Length of Stay in Days'] = (df['Discharge Date'] - df['Date of Admission']).dt.days

# Reorder columns: insert 'Length of Stay' before 'Target'
target_index = df.columns.get_loc('Test Results')
cols = list(df.columns)
# Move 'Length of Stay' to the position before 'Target'
cols.insert(target_index, cols.pop(cols.index('Length of Stay in Days')))
df = df[cols]

# drop 'Discharge Date' and 'Date of Admission' columns
df.drop(columns=['Discharge Date', 'Date of Admission'], inplace=True)
df.info()


In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
#Checking the percentage of the missing data
pd.set_option('display.max_rows', None)
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Feature': df.columns, 'Missing Percentage': missing_percentage})
print(missing_df)

In [ ]:
df.duplicated().sum()

In [ ]:
look_for_outliers(df)

In [ ]:
# List of numerical features
numerical_features = [col for col in df.select_dtypes(include=['int64', 'float64']).columns if col != 'Age']

# Apply the function to handle outliers
df = handle_outliers_iqr(df, numerical_features)

In [ ]:
look_for_outliers(df)

In [ ]:
# Drop ID and Name
df.drop(columns=['ID', 'Name', 'Room Number'], inplace=True)

In [ ]:
pd.set_option('display.max_rows', None)  #this line to show all of the records
df_dtypes =pd.DataFrame({"Feature": df.columns, "Data Type": df.dtypes})
print(df_dtypes)
pd.reset_option('display.max_rows')

In [ ]:
numerical_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
numerical_features = [col for col in numerical_features if col != ['Test Results', 'Age']]

df, skewed_features, transformation_details = treat_skewness(df, numerical_features)

In [ ]:
df, scaled_cols = scale_features(df, target_col='Test Results', scaler_type='standard')

In [ ]:
df.head()

In [ ]:
X_encoded, y_encoded = encode_features(df, target_col="Test Results")
df = pd.concat([X_encoded, y_encoded.rename("Test Results")], axis=1)

In [ ]:
pd.set_option('display.max_rows', None)  #this line to show all of the records
df_dtypes = pd.DataFrame({"Feature": df.columns, "Data Type": df.dtypes})
print(df_dtypes)
pd.reset_option('display.max_rows')

In [ ]:
numeric_desc = df.describe()
print("Descriptive Statistics (Numeric Columns):")
print(numeric_desc)

In [ ]:
df.info()

In [ ]:
print(df.columns.tolist())

Statistical Analysis after cleaning

In [ ]:
# Histograms for numerical features
numeric_cols = df.select_dtypes(include=[np.number]).columns

df[numeric_cols].hist(figsize=(16, 12), bins=30, color='skyblue', edgecolor='black')
plt.suptitle('Histograms of Numerical Features', fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
# Confusion Heat Map
plt.figure(figsize=(25, 25))
sns.heatmap(df.corr(), 
            annot=True, 
            cmap='hsv', 
            fmt='.3f', 
            linewidths=2)

plt.title('Correlation Heatmap (Including Encoded Categorical Features)', fontsize=16)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# | Absolute Correlation Value | Strength                          |
# | -------------------------- | --------------------------------- |
# | 0.0 - 0.1                  | Negligible                        |
# | 0.1 - 0.3                  | Weak                              |
# | 0.3 - 0.5                  | Moderate                          |
# | 0.5 - 0.7                  | Strong                            |
# | 0.7 - 0.9                  | Very Strong                       |
# | 0.9 - 1.0                  | Extremely High / Likely Redundant |


def get_correlation_table(df):
    # Compute correlation matrix
    target='Test Results'
    correlation_matrix = df.corr()

    # Extract only correlations with the target, excluding the target itself
    target_corr = correlation_matrix[target].drop(labels=[target])

    # Convert to a clean DataFrame
    corr_table = pd.DataFrame({
        'Feature': target_corr.index,
        'Correlation with ' + target: target_corr.values
    }).sort_values(by='Correlation with ' + target, ascending=False).reset_index(drop=True)

    return corr_table


get_correlation_table(df)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Replace these with actual column names from your dataset
feature_col = 'Age'      # Any categorical or discrete numeric feature
target_col = 'Test Results'    # Your class label column

# Check the columns exist
assert feature_col in df.columns and target_col in df.columns, "Check your column names"

# Create the crosstab and plot
plt.figure(figsize=(25, 8))
pd.crosstab(df[feature_col], df[target_col]).plot(kind='bar',
                                                figsize=(25, 8), 
                                                color=['gold', 'brown'])
plt.title(f'{target_col.capitalize()} Frequency for {feature_col.capitalize()}')
plt.xlabel(feature_col.capitalize())
plt.ylabel('Frequency')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# creates a grid of plots that shows pairwise relationships between all numeric columns in df.
sns.pairplot(data=df)

In [ ]:
# save dataframe to csv
df.to_csv("data/processed_data.csv", index=False)
print("[INFO] Preprocessed dataset saved as 'preprocessed_data.csv'")

PCA

In [ ]:
from utils.PCA import apply_pca
from utils.scaling import scale_features
from sklearn.feature_selection import VarianceThreshold
import pandas as pd
from sklearn.decomposition import PCA

In [ ]:
df = pd.read_csv("data/processed_data.csv")

#Select numeric colums
numeric_cols = df.select_dtypes(include=['number']).columns
print("[INFO] Numeric columns:", list(numeric_cols))

#Scale numeric features
X_scaled, _ = scale_features(df[numeric_cols], scaler_type='standard')

#Feature selection - remove low variance features
selector = VarianceThreshold(threshold=0.1)
X_highvar = selector.fit_transform(X_scaled)

#Apply PCA
X_pca, pca = apply_pca(X_highvar, variance_threshold=0.99)

#Output
print(f"Features after variance threshold: {X_highvar.shape[1]}")
print(f"Selected components: {X_pca.shape[1]}")
print("Top components:")
for i, ratio in enumerate(pca.explained_variance_ratio_[:5]):
    print(f"PC{i+1}: {ratio:.1%}")

In [ ]:
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o')
plt.title("Cumulative Explained Variance")
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Variance")
plt.grid(True)
plt.show()

In [ ]:
# Define PCA column names
pca_columns = [f'PC{i+1}' for i in range(X_pca.shape[1])]
pca_data = pd.DataFrame(X_pca, columns=pca_columns)

# Read labels and reset index
y = pd.read_csv("data/processed_data.csv")['Test Results'].reset_index(drop=True)

# Combine PCA features and labels
df_pca = pd.concat([pca_data, y], axis=1)

# Save to CSV
df_pca.to_csv('data/PCA_data.csv', index=False)
print("[INFO] PCA-transformed dataset saved as 'PCA_data.csv' with Target 'Test Results' ")

Training Models Without PCA

In [ ]:
from models.randomforest import random_forest_with_grid
from models.svm import (
    linear_svm_with_grid, rbf_svm_with_pso
)

In [ ]:
# read preprocessed data
df_cleaned = pd.read_csv("data/processed_data.csv")

x = df_cleaned.drop(columns=['Test Results'])
y = df_cleaned['Test Results']

# split data intro train, test and validation sets
X_train, X_temp, y_train, y_temp = train_test_split(x, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Training set size: {X_train.shape[0]}")
print(f"Validation set size: {X_val.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

Random Forest

In [ ]:
# RANDOM FOREST MODEL
print("[INFO] Training Random Forest model...")
rf_model = random_forest_with_grid(X_train, y_train, X_val, y_val, X_test, y_test)

SVM

In [ ]:
# LINEAR SVM MODEL
print("[INFO] Training Linear SVM model...")
svm_model = linear_svm_with_grid(X_train, y_train, X_val, y_val, X_test, y_test)
print(svm_model)

In [ ]:
# # RBF SVM MODEL
# print("[INFO] Training RBF SVM model...")
# rbf_svm_model = rbf_svm_with_pso(X_train, X_val, X_test, y_train, y_val, y_test)
# print(rbf_svm_model)

MLP

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from models.MLP import run_hyperparameter_tuning, train_final_model,train_default_model, evaluate_model

In [ ]:
print("=== Training without hyperparameter tuning ===")
default_model, _ = train_default_model(X_train, y_train, X_val, y_val, input_shape=X_train.shape[1])
evaluate_model(default_model, X_test, y_test)

In [ ]:
print("\n=== Training with hyperparameter tuning ===")
tuner = run_hyperparameter_tuning(X_train, y_train, X_val, y_val, input_shape=X_train.shape[1])
tuned_model, _ = train_final_model(tuner, X_train, y_train, X_val, y_val)
evaluate_model(tuned_model, X_test, y_test)

Naive Bayes

In [ ]:
nb_model = naive_bayes_classifier(X_train, X_val, X_test, y_train, y_val, y_test)

In [ ]:
nb_tuned_model = naive_bayes_with_grid_search(
    X_train, X_val, X_test, y_train, y_val, y_test
)

Training Models With PCA

In [ ]:
# Read PCA data (features)
pca_data = pd.read_csv('data/PCA_data.csv')

# Print for sanity check
print(df_pca.columns)
print(df_pca.head())
print(df_pca.shape)

# Separate features and target
x = df_pca.drop(columns=['Test Results'])
y = df_pca['Test Results']

In [ ]:
# Split into training and temp (for val + test) (training 70%, temp 30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    x, y, test_size=0.3, random_state=42, stratify=y
)

# Split temp into validation and test sets (validation 15%, test 15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# Print sizes
print(f"Training set size: {X_train.shape[0]}")
print(f"Validation set size: {X_val.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

Random Forest

In [ ]:
# RANDOM FOREST MODEL
print("[INFO] Training Random Forest model...")
rf_model = random_forest_with_grid(X_train, y_train, X_val, y_val, X_test, y_test)

SVM

In [ ]:
# LINEAR SVM MODEL
print("[INFO] Training Linear SVM model...")
svm_model = linear_svm_with_grid(X_train, y_train, X_val, y_val, X_test, y_test)

In [ ]:
# # RBF SVM MODEL
# print("[INFO] Training RBF SVM model...")
# rbf_svm_model = rbf_svm_with_pso(X_train, X_val, X_test, y_train, y_val, y_test)


Naive Bayes

In [ ]:
nb_model = naive_bayes_classifier(X_train, X_val, X_test, y_train, y_val, y_test)

In [ ]:
nb_tuned_model = naive_bayes_with_grid_search(
    X_train, X_val, X_test, y_train, y_val, y_test
)

MLP

In [ ]:
print("=== Training without hyperparameter tuning ===")
default_model, _ = train_default_model(X_train, y_train, X_val, y_val, input_shape=X_train.shape[1])
evaluate_model(default_model, X_test, y_test)

In [ ]:
print("\n=== Training with hyperparameter tuning ===")
tuner = run_hyperparameter_tuning(X_train, y_train, X_val, y_val, input_shape=X_train.shape[1])
tuned_model, _ = train_final_model(tuner, X_train, y_train, X_val, y_val)
evaluate_model(tuned_model, X_test, y_test)